## **Phase 2 (Peer Review)**

#### Project Information

**Research Question:** How can regional and weather factors like cloud cover and elevation influence the performance of drivers across teams, possibly highlighting team-specific specializations?

**Dataset Description (F1 Race Data)**
> https://openf1.org/docs/

<sup><sub>**Notes about the dataset:** The first dataset includes historical F1 race data able to be filtered by driver, race, or date. We can compare different teams and individuals and how their lap times, their pit duration, and any other relevant podium-seeking information varies. Via the OpenF1 API (https://github.com/br-g/openf1), we get access to car telemetry (https://openf1.org/docs/#car-data), lap times (https://openf1.org/docs/#laps), race positions (https://openf1.org/docs/#position), pit stops (https://openf1.org/docs/#pit), and race control (https://openf1.org/docs/#race-control) from all races, teams, and racers. Access is limited to historical data, meaning the 2023 season onward. The weather information (https://openf1.org/docs/#weather) is quite limited excluding factors like cloud cover and elevation, which is why we are looking at Open Meteo API to supplement OpenF1. On OpenF1, there are also generous rate limits of 3 requests per second or 30 requests per minute (https://openf1.org/#faq). OpenF1 is a free, community-run API, so its data is generally reliable for stats because it's a large community, but it isn't an official source of record. Veritably, OpenF1 is directly based on F1 timing/telemetry data making it suitable for academic-style preparatory work, which is our purpose.</sub></sup>

**Dataset Description (Weather Data)**
> https://open-meteo.com/en/docs/historical-weather-api

<sup><sub>**Notes about the dataset:** The second dataset includes realtime, historical weather data that can be used to supplement our analysis of race results. The main access limitation with the Open Meteo API is the spatial resolution, having grid cells ranging from 11 km to 25 km in size. The specific set we’d end up using is likely the ERA5 (https://doi.org/10.24381/cds.adbb2d47) for our use case. These reports are hourly and constructed from data 1950 and onward. Open Meteo, construction described at this site (https://doi.org/10.5281/ZENODO.7970649) may lose some reliability in how its data is built off of reanalysis of various sources (weather stations, aircrafts, buoys, radar, etc.) rather than a point observation.</sub></sup>

#### Data cleaning

##### Libraries

In [39]:
# installing necessary libraries
%pip install seaborn pandas duckdb requests numpy

Note: you may need to restart the kernel to use updated packages.


In [40]:
# loading necessary libraries and packages to handle our data
import requests # to extract from datasets
import pandas as pd # to handle the dataframes
import duckdb # to clean the data
from datetime import datetime # to handle the date time format

Note for above code chunk(s), will be modifying and editing as we go to ensure we have what we need to handle our data.

##### Race Session Import and Cleansing

In [41]:
# code OpenF1 API informs us to use to import
from urllib.request import urlopen
import json
response = urlopen('https://api.openf1.org/v1/sessions')
data = json.loads(response.read().decode('utf-8'))
temp_races_df = pd.DataFrame(data) # last line of provided code from API

For above code, this was provided by OpenF1 API in (https://openf1.org/docs) for information on how to import/scrape.

In [48]:
temp_races_df # checking the column headers
temp_races_df['session_type'].unique() # checking to see what the different types are to make sure we don't truncate

# selecting the wanted columns and races
races_df = duckdb.sql("SELECT session_key, session_type, session_name, date_start, date_end, circuit_short_name, country_name, location, year \
                      FROM temp_races_df WHERE ((session_type = 'Race' OR session_type = 'Qualifying') AND is_cancelled = False) \
                      AND year < 2026").df()
races_df # checking the product and modifying sql as necessary to prune canceled races and races from 2026 (ongoing) season
races_df['date_start'] = pd.to_datetime(races_df['date_start'], format='mixed') # reformat dates
races_df['date_end'] = pd.to_datetime(races_df['date_end'], format='mixed')
# wanting to uniform the time zone and make it more easily comprehensible for human reading
races_df['date_start'] = races_df['date_start'].dt.tz_convert("EST").dt.tz_localize(None)
races_df['date_end'] = races_df['date_end'].dt.tz_convert("EST").dt.tz_localize(None)
races_df # checking date time

,session_key,session_type,session_name,date_start,date_end,circuit_short_name,country_name,location,year
0,7768,Qualifying,Qualifying,2023-03-04 10:00:00,2023-03-04 11:00:00,Sakhir,Bahrain,Sakhir,2023
1,7953,Race,Race,2023-03-05 10:00:00,2023-03-05 12:00:00,Sakhir,Bahrain,Sakhir,2023
2,7775,Qualifying,Qualifying,2023-03-18 12:00:00,2023-03-18 13:00:00,Jeddah,Saudi Arabia,Jeddah,2023
3,7779,Race,Race,2023-03-19 12:00:00,2023-03-19 14:00:00,Jeddah,Saudi Arabia,Jeddah,2023
4,7783,Qualifying,Qualifying,2023-04-01 01:00:00,2023-04-01 02:00:00,Melbourne,Australia,Melbourne,2023
...,...,...,...,...,...,...,...,...,...
171,9845,Race,Sprint,2025-11-29 09:00:00,2025-11-29 10:00:00,Lusail,Qatar,Lusail,2025
172,9846,Qualifying,Qualifying,2025-11-29 13:00:00,2025-11-29 14:00:00,Lusail,Qatar,Lusail,2025
173,9850,Race,Race,2025-11-30 11:00:00,2025-11-30 13:00:00,Lusail,Qatar,Lusail,2025
174,9835,Qualifying,Qualifying,2025-12-06 09:00:00,2025-12-06 10:00:00,Yas Marina Circuit,United Arab Emirates,Yas Island,2025


Here, we have cleaned a DataFrame to contain the dates and times for races and the races that qualify the teams to participate in the larger race for narrowing down the time and date for the weather conditions. We also made sure to maintain the specific names of the locations to further assist us when looping in information from Open Meteo. We found the session types listed were `[Practice, Qualifying, Race]` and chose to disregard Practices since teams use these for practice and performance may not be up to par; in other words, since the teams are warming up or practicing on the track, the information consumed may not be as relevant to driver overall performance when representing their team name. One other important thing to note is that we filtered out the race dates from 2026. Since the 2026 F1 season is still ongoing, these races will lack information about the performance of teams and their drivers. We prefer to handle a constantly changing dataset, so we decided to stick with 2023, 2024, and 2025 where all the data is set.

##### Drivers and Their Teams Import and Cleansing

In [44]:
# code OpenF1 API informs us to use to import
response = urlopen('https://api.openf1.org/v1/drivers')
data = json.loads(response.read().decode('utf-8'))
temp_drivers_df = pd.DataFrame(data) # last line of provided code from API

For above code, this was provided by OpenF1 API in (https://openf1.org/docs) for information on how to import/scrape.

In [60]:
temp_drivers_df # checking the column headers
temp_drivers_df['broadcast_name'].unique() # seeing who is really driver because original is 10k lines

drivers_df = duckdb.sql("SELECT DISTINCT r.year, d.driver_number, d.full_name, d.name_acronym, d.team_name, d.country_code \
                      FROM temp_drivers_df d JOIN races_df r ON r.session_key = d.session_key ORDER BY d.full_name, d.team_name, r.year").df()
drivers_df

,year,driver_number,full_name,name_acronym,team_name,country_code
0,2023,23,Alexander ALBON,ALB,Williams,THA
1,2024,23,Alexander ALBON,ALB,Williams,THA
2,2025,23,Alexander ALBON,ALB,Williams,NaN
3,2023,55,Carlos SAINZ,SAI,Ferrari,ESP
4,2024,55,Carlos SAINZ,SAI,Ferrari,ESP
...,...,...,...,...,...,...
65,2024,22,Yuki TSUNODA,TSU,RB,JPN
66,2025,22,Yuki TSUNODA,TSU,Racing Bulls,NaN
67,2025,22,Yuki TSUNODA,TSU,Red Bull Racing,NaN
68,2023,24,ZHOU Guanyu,ZHO,Alfa Romeo,CHN


Here, we have cleaned a DataFrame to contain the names, team names, years, and countries of the drivers. One thing that makes this dataset difficult to handle is how the same driver might have different driver numbers or teams or country associations year to year. This is something we will have to deal with as we analyze the data. I decided to join the race date information to help add understanding to the team associations. Since team associations might shift year to year, this helps narrow it down. Additionally, joining on the session keys helps ensure we aren't including information from practices since the session keys from `races_df` would be pruned down by date, race cancellation, etc. We choose to focus on `session_key` and not `meeting_key` since `meeting_key` is specific to special meets and chunks entire weekends together. `session_key` keeps each individual track session separate, which is far more specific.

##### Weather Information Import and Cleansing

In [ ]:
races_df['circuit_short_name'].unique() # to print out the entire full list of circuit names scrollable

# research each circuit name for latitude and longitude
circuit_coords = {
    # OpenF1 circuit_short_name: (latitude, longitude) # full name
    "Sakhir": (26.0325, 50.5106),   # Bahrain International Circuit
    "Jeddah": (21.6319, 39.1044),   # Jeddah Corniche Circuit
    "Melbourne": (-37.8497, 144.9680),   # Albert Park
    "Baku": (40.3725, 49.8533),   # Baku City Circuit
    "Miami": (25.9581, -80.2389),   # Miami International Autodrome
    "Monte Carlo": (43.7347, 7.4206),   # Circuit de Monaco
    "Catalunya": (41.5700, 2.2611),   # Circuit de Barcelona-Catalunya
    "Montreal": (45.5017, -73.5228),   # Circuit Gilles Villeneuve
    "Spielberg": (47.2197, 14.7647),   # Red Bull Ring
    "Silverstone": (52.0786, -1.0169),   # Silverstone Circuit
    "Hungaroring": (47.5789, 19.2486),   # Hungaroring
    "Spa-Francorchamps": (50.4372, 5.9714),   # Circuit de Spa-Francorchamps
    "Zandvoort": (52.3888, 4.5409),   # Circuit Zandvoort
    "Monza": (45.6156, 9.2811),   # Autodromo Nazionale Monza
    "Singapore": (1.2914, 103.8640),   # Marina Bay Street Circuit
    "Suzuka": (34.8431, 136.5410),   # Suzuka International Racing Course
    "Lusail": (25.4900, 51.4542),   # Lusail International Circuit
    "Austin": (30.1328, -97.6411),   # Circuit of the Americas
    "Mexico City": (19.4042, -99.0907),   # Autódromo Hermanos Rodríguez
    "Interlagos": (-23.7036, -46.6997),   # Autódromo José Carlos Pace
    "Las Vegas": (36.1147, -115.1728),   # Las Vegas Strip Circuit
    "Yas Marina Circuit": (24.4672, 54.6031),   # Yas Marina Circuit
    "Shanghai": (31.3389, 121.2197),   # Shanghai International Circuit
    "Imola": (44.3439, 11.7167),   # Autodromo Enzo e Dino Ferrari
}

coords_df = pd.DataFrame.from_dict(circuit_coords, orient='index', columns=['latitude', 'longitude']).rename_axis("circuit_short_name").reset_index()
coords_df # ensure it looks correct

# prepping to input information to Open Meteo in their specified format
# latitude, longitude, , timezone, start date, end date
temp_races_df['date_start'] = races_df['date_start'].dt.strftime("%Y-%m-%d")
temp_races_df['date_end'] = races_df['date_end'].dt.strftime("%Y-%m-%d")
openmeteo_input_df = duckdb.sql("SELECT c.latitude, c.longitude, r.date_start, r.date_end FROM coords_df c JOIN temp_races_df r \
                                ON c.circuit_short_name = r.circuit_short_name WHERE r.date_start != 'NaN' AND r.date_end != 'NaN'").df()
timezone_filler = [',EST'] * 176
openmeteo_input_df.insert(loc=2, column='timezone', value=timezone_filler)
openmeteo_input_df # print to see if format good

openmeteo_input_df.to_csv('openmeteo_input.csv', index=False) # exporting the format
openmeteo_input_df['hour'] = races_df['date_start'].dr.strftime("")

Above, we modified our existing datasets and information to be able to input it to Open Meteo more easily.

In [ ]:
# code provided to us to input for importing data by the Open Meteo website
%pip install openmeteo-requests
%pip install requests-cache retry-requests numpy pandas

import openmeteo_requests
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

In [ ]:
# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 26.0325,
	"longitude": 50.5106,
	"start_date": "2023-03-04 00:00:00",
	"end_date": "2023-03-04 01:00:00",
	"hourly": ["temperature_2m", "precipitation", "cloud_cover_low", "cloud_cover_high", "cloud_cover_mid", "cloud_cover", "surface_pressure", "wind_speed_10m", "wind_direction_10m", "wind_gusts_10m", "wind_direction_100m", "wind_speed_100m"],
	"timezone": "EST",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(1).ValuesAsNumpy()
hourly_cloud_cover_low = hourly.Variables(2).ValuesAsNumpy()
hourly_cloud_cover_high = hourly.Variables(3).ValuesAsNumpy()
hourly_cloud_cover_mid = hourly.Variables(4).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(5).ValuesAsNumpy()
hourly_surface_pressure = hourly.Variables(6).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(7).ValuesAsNumpy()
hourly_wind_direction_10m = hourly.Variables(8).ValuesAsNumpy()
hourly_wind_gusts_10m = hourly.Variables(9).ValuesAsNumpy()
hourly_wind_direction_100m = hourly.Variables(10).ValuesAsNumpy()
hourly_wind_speed_100m = hourly.Variables(11).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	).tz_convert(response.Timezone().decode())
}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["precipitation"] = hourly_precipitation
hourly_data["cloud_cover_low"] = hourly_cloud_cover_low
hourly_data["cloud_cover_high"] = hourly_cloud_cover_high
hourly_data["cloud_cover_mid"] = hourly_cloud_cover_mid
hourly_data["cloud_cover"] = hourly_cloud_cover
hourly_data["surface_pressure"] = hourly_surface_pressure
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["wind_direction_10m"] = hourly_wind_direction_10m
hourly_data["wind_gusts_10m"] = hourly_wind_gusts_10m
hourly_data["wind_direction_100m"] = hourly_wind_direction_100m
hourly_data["wind_speed_100m"] = hourly_wind_speed_100m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n


Coordinates: 26.045692443847656°N 50.514225006103516°E
Elevation: 8.0 m asl
Timezone: b'EST'b'GMT-5'
Timezone difference to GMT+0: -18000s

Hourly data
                         date  temperature_2m  precipitation  cloud_cover_low  \
0  2023-03-04 00:00:00-05:00       18.650000            0.0              0.0   
1  2023-03-04 01:00:00-05:00       22.950001            0.0              0.0   
2  2023-03-04 02:00:00-05:00       25.500000            0.0              0.0   
3  2023-03-04 03:00:00-05:00       27.250000            0.0              0.0   
4  2023-03-04 04:00:00-05:00       28.799999            0.0              0.0   
5  2023-03-04 05:00:00-05:00       29.500000            0.0              0.0   
6  2023-03-04 06:00:00-05:00       29.850000            0.0              0.0   
7  2023-03-04 07:00:00-05:00       29.299999            0.0              0.0   
8  2023-03-04 08:00:00-05:00       27.700001            0.0              0.0   
9  2023-03-04 09:00:00-05:00       25.350000  

Open Meteo API provided us with specific Python we could use to extract our data. To see the code Open Meteo provided, click [here](https://open-meteo.com/en/docs/historical-weather-api?location_mode=csv_coordinates&csv_coordinates=26.0325,50.5106,,EST,2023-03-04,2023-03-04%0A26.0325,50.5106,,EST,2023-03-05,2023-03-05%0A26.0325,50.5106,,EST,2023-03-18,2023-03-18%0A26.0325,50.5106,,EST,2023-03-19,2023-03-19%0A26.0325,50.5106,,EST,2023-04-01,2023-04-01%0A26.0325,50.5106,,EST,2023-04-02,2023-04-02%0A26.0325,50.5106,,EST,2023-04-28,2023-04-28%0A26.0325,50.5106,,EST,2023-04-29,2023-04-29%0A21.6319,39.1044,,EST,2023-04-29,2023-04-29%0A21.6319,39.1044,,EST,2023-04-30,2023-04-30%0A21.6319,39.1044,,EST,2023-05-06,2023-05-06%0A21.6319,39.1044,,EST,2023-05-07,2023-05-07%0A21.6319,39.1044,,EST,2023-05-27,2023-05-27%0A-37.8497,144.968,,EST,2023-05-28,2023-05-28%0A-37.8497,144.968,,EST,2023-06-03,2023-06-03%0A-37.8497,144.968,,EST,2023-06-04,2023-06-04%0A-37.8497,144.968,,EST,2023-06-17,2023-06-17%0A-37.8497,144.968,,EST,2023-06-18,2023-06-18%0A40.3725,49.8533,,EST,2023-06-30,2023-06-30%0A40.3725,49.8533,,EST,2023-07-01,2023-07-01%0A40.3725,49.8533,,EST,2023-07-01,2023-07-01%0A40.3725,49.8533,,EST,2023-07-02,2023-07-02%0A40.3725,49.8533,,EST,2023-07-08,2023-07-08%0A25.9581,-80.2389,,EST,2023-07-09,2023-07-09%0A25.9581,-80.2389,,EST,2023-07-22,2023-07-22%0A25.9581,-80.2389,,EST,2023-07-23,2023-07-23%0A25.9581,-80.2389,,EST,2023-07-28,2023-07-28%0A25.9581,-80.2389,,EST,2023-07-29,2023-07-29%0A44.3439,11.7167,,EST,2023-07-29,2023-07-29%0A44.3439,11.7167,,EST,2023-07-30,2023-07-30%0A44.3439,11.7167,,EST,2023-08-26,2023-08-26%0A44.3439,11.7167,,EST,2023-08-27,2023-08-27%0A44.3439,11.7167,,EST,2023-09-02,2023-09-02%0A43.7347,7.4206,,EST,2023-09-03,2023-09-03%0A43.7347,7.4206,,EST,2023-09-16,2023-09-16%0A43.7347,7.4206,,EST,2023-09-17,2023-09-17%0A43.7347,7.4206,,EST,2023-09-23,2023-09-23%0A43.7347,7.4206,,EST,2023-09-24,2023-09-24%0A41.57,2.2611,,EST,2023-10-06,2023-10-06%0A41.57,2.2611,,EST,2023-10-07,2023-10-07%0A41.57,2.2611,,EST,2023-10-07,2023-10-07%0A41.57,2.2611,,EST,2023-10-08,2023-10-08%0A41.57,2.2611,,EST,2023-10-20,2023-10-20%0A45.5017,-73.5228,,EST,2023-10-21,2023-10-21%0A45.5017,-73.5228,,EST,2023-10-21,2023-10-21%0A45.5017,-73.5228,,EST,2023-10-22,2023-10-22%0A45.5017,-73.5228,,EST,2023-10-28,2023-10-28%0A45.5017,-73.5228,,EST,2023-10-29,2023-10-29%0A47.2197,14.7647,,EST,2023-11-03,2023-11-03%0A47.2197,14.7647,,EST,2023-11-04,2023-11-04%0A47.2197,14.7647,,EST,2023-11-04,2023-11-04%0A47.2197,14.7647,,EST,2023-11-05,2023-11-05%0A47.2197,14.7647,,EST,2023-11-18,2023-11-18%0A52.0786,-1.0169,,EST,2023-11-19,2023-11-19%0A52.0786,-1.0169,,EST,2023-11-25,2023-11-25%0A52.0786,-1.0169,,EST,2023-11-26,2023-11-26%0A52.0786,-1.0169,,EST,2024-03-01,2024-03-01%0A52.0786,-1.0169,,EST,2024-03-02,2024-03-02%0A47.5789,19.2486,,EST,2024-03-08,2024-03-08%0A47.5789,19.2486,,EST,2024-03-09,2024-03-09%0A47.5789,19.2486,,EST,2024-03-23,2024-03-23%0A47.5789,19.2486,,EST,2024-03-23,2024-03-24%0A47.5789,19.2486,,EST,2024-04-06,2024-04-06%0A50.4372,5.9714,,EST,2024-04-07,2024-04-07%0A50.4372,5.9714,,EST,2024-04-19,2024-04-19%0A50.4372,5.9714,,EST,2024-04-19,2024-04-19%0A50.4372,5.9714,,EST,2024-04-20,2024-04-20%0A50.4372,5.9714,,EST,2024-04-21,2024-04-21%0A52.3888,4.5409,,EST,2024-05-03,2024-05-03%0A52.3888,4.5409,,EST,2024-05-04,2024-05-04%0A52.3888,4.5409,,EST,2024-05-04,2024-05-04%0A52.3888,4.5409,,EST,2024-05-05,2024-05-05%0A52.3888,4.5409,,EST,2024-05-18,2024-05-18%0A45.6156,9.2811,,EST,2024-05-19,2024-05-19%0A45.6156,9.2811,,EST,2024-05-25,2024-05-25%0A45.6156,9.2811,,EST,2024-05-26,2024-05-26%0A45.6156,9.2811,,EST,2024-06-08,2024-06-08%0A45.6156,9.2811,,EST,2024-06-09,2024-06-09%0A1.2914,103.864,,EST,2024-06-22,2024-06-22%0A1.2914,103.864,,EST,2024-06-23,2024-06-23%0A1.2914,103.864,,EST,2024-06-28,2024-06-28%0A1.2914,103.864,,EST,2024-06-29,2024-06-29%0A1.2914,103.864,,EST,2024-06-29,2024-06-29%0A34.8431,136.541,,EST,2024-06-30,2024-06-30%0A34.8431,136.541,,EST,2024-07-06,2024-07-06%0A34.8431,136.541,,EST,2024-07-07,2024-07-07%0A34.8431,136.541,,EST,2024-07-20,2024-07-20%0A34.8431,136.541,,EST,2024-07-21,2024-07-21%0A25.49,51.4542,,EST,2024-07-27,2024-07-27%0A25.49,51.4542,,EST,2024-07-28,2024-07-28%0A25.49,51.4542,,EST,2024-08-24,2024-08-24%0A25.49,51.4542,,EST,2024-08-25,2024-08-25%0A25.49,51.4542,,EST,2024-08-31,2024-08-31%0A30.1328,-97.6411,,EST,2024-09-01,2024-09-01%0A30.1328,-97.6411,,EST,2024-09-14,2024-09-14%0A30.1328,-97.6411,,EST,2024-09-15,2024-09-15%0A30.1328,-97.6411,,EST,2024-09-21,2024-09-21%0A30.1328,-97.6411,,EST,2024-09-22,2024-09-22%0A19.4042,-99.0907,,EST,2024-10-18,2024-10-18%0A19.4042,-99.0907,,EST,2024-10-19,2024-10-19%0A19.4042,-99.0907,,EST,2024-10-19,2024-10-19%0A19.4042,-99.0907,,EST,2024-10-20,2024-10-20%0A19.4042,-99.0907,,EST,2024-10-26,2024-10-26%0A-23.7036,-46.6997,,EST,2024-10-27,2024-10-27%0A-23.7036,-46.6997,,EST,2024-11-01,2024-11-01%0A-23.7036,-46.6997,,EST,2024-11-02,2024-11-02%0A-23.7036,-46.6997,,EST,2024-11-03,2024-11-03%0A-23.7036,-46.6997,,EST,2024-11-03,2024-11-03%0A36.1147,-115.1728,,EST,2024-11-23,2024-11-23%0A36.1147,-115.1728,,EST,2024-11-24,2024-11-24%0A36.1147,-115.1728,,EST,2024-11-29,2024-11-29%0A36.1147,-115.1728,,EST,2024-11-30,2024-11-30%0A36.1147,-115.1728,,EST,2024-11-30,2024-11-30%0A24.4672,54.6031,,EST,2024-12-01,2024-12-01%0A24.4672,54.6031,,EST,2024-12-07,2024-12-07%0A24.4672,54.6031,,EST,2024-12-08,2024-12-08%0A24.4672,54.6031,,EST,2025-03-15,2025-03-15%0A24.4672,54.6031,,EST,2025-03-15,2025-03-16%0A26.0325,50.5106,,EST,2025-03-21,2025-03-21%0A26.0325,50.5106,,EST,2025-03-21,2025-03-21%0A26.0325,50.5106,,EST,2025-03-22,2025-03-22%0A26.0325,50.5106,,EST,2025-03-23,2025-03-23%0A26.0325,50.5106,,EST,2025-04-05,2025-04-05%0A26.0325,50.5106,,EST,2025-04-06,2025-04-06%0A26.0325,50.5106,,EST,2025-04-12,2025-04-12%0A26.0325,50.5106,,EST,2025-04-13,2025-04-13%0A21.6319,39.1044,,EST,2025-04-19,2025-04-19%0A21.6319,39.1044,,EST,2025-04-20,2025-04-20%0A21.6319,39.1044,,EST,2025-05-02,2025-05-02%0A21.6319,39.1044,,EST,2025-05-03,2025-05-03%0A21.6319,39.1044,,EST,2025-05-03,2025-05-03%0A-37.8497,144.968,,EST,2025-05-04,2025-05-04%0A-37.8497,144.968,,EST,2025-05-17,2025-05-17%0A-37.8497,144.968,,EST,2025-05-18,2025-05-18%0A-37.8497,144.968,,EST,2025-05-24,2025-05-24%0A-37.8497,144.968,,EST,2025-05-25,2025-05-25%0A34.8431,136.541,,EST,2025-05-31,2025-05-31%0A34.8431,136.541,,EST,2025-06-01,2025-06-01%0A34.8431,136.541,,EST,2025-06-14,2025-06-14%0A34.8431,136.541,,EST,2025-06-15,2025-06-15%0A34.8431,136.541,,EST,2025-06-28,2025-06-28%0A31.3389,121.2197,,EST,2025-06-29,2025-06-29%0A31.3389,121.2197,,EST,2025-07-05,2025-07-05%0A31.3389,121.2197,,EST,2025-07-06,2025-07-06%0A31.3389,121.2197,,EST,2025-07-25,2025-07-25%0A31.3389,121.2197,,EST,2025-07-26,2025-07-26%0A25.9581,-80.2389,,EST,2025-07-26,2025-07-26%0A25.9581,-80.2389,,EST,2025-07-27,2025-07-27%0A25.9581,-80.2389,,EST,2025-08-02,2025-08-02%0A25.9581,-80.2389,,EST,2025-08-03,2025-08-03%0A25.9581,-80.2389,,EST,2025-08-30,2025-08-30%0A44.3439,11.7167,,EST,2025-08-31,2025-08-31%0A44.3439,11.7167,,EST,2025-09-06,2025-09-06%0A44.3439,11.7167,,EST,2025-09-07,2025-09-07%0A44.3439,11.7167,,EST,2025-09-20,2025-09-20%0A44.3439,11.7167,,EST,2025-09-21,2025-09-21%0A43.7347,7.4206,,EST,2025-10-04,2025-10-04%0A43.7347,7.4206,,EST,2025-10-05,2025-10-05%0A43.7347,7.4206,,EST,2025-10-17,2025-10-17%0A43.7347,7.4206,,EST,2025-10-18,2025-10-18%0A43.7347,7.4206,,EST,2025-10-18,2025-10-18%0A45.5017,-73.5228,,EST,2025-10-19,2025-10-19%0A45.5017,-73.5228,,EST,2025-10-25,2025-10-25%0A45.5017,-73.5228,,EST,2025-10-26,2025-10-26%0A45.5017,-73.5228,,EST,2025-11-07,2025-11-07%0A45.5017,-73.5228,,EST,2025-11-08,2025-11-08%0A41.57,2.2611,,EST,2025-11-08,2025-11-08%0A41.57,2.2611,,EST,2025-11-09,2025-11-09%0A41.57,2.2611,,EST,2025-11-21,2025-11-22%0A41.57,2.2611,,EST,2025-11-22,2025-11-23%0A41.57,2.2611,,EST,2025-11-28,2025-11-28%0A47.2197,14.7647,,EST,2025-11-29,2025-11-29%0A47.2197,14.7647,,EST,2025-11-29,2025-11-29%0A47.2197,14.7647,,EST,2025-11-30,2025-11-30%0A47.2197,14.7647,,EST,2025-12-06,2025-12-06%0A47.2197,14.7647,,EST,2025-12-07,2025-12-07&start_date=2023-01-01&end_date=2025-12-30&hourly=temperature_2m,precipitation,cloud_cover_low,cloud_cover_high,cloud_cover_mid,cloud_cover,surface_pressure,wind_speed_10m,wind_direction_10m,wind_gusts_10m,wind_direction_100m,wind_speed_100m).